### **.pkl** model load and information checking

In [ ]:
MODEL_FOLDER = "../ThirdParty/Strategies/models/"

In [ ]:
import joblib
pkl_model_path = MODEL_FOLDER + "rf_model.pkl"
rf_model = joblib.load(pkl_model_path)

# basic info
print("n_features_in:", rf_model.n_features_in_)
print("n_estimators:", rf_model.n_estimators)
print("max_depth:", rf_model.max_depth)
print("min_samples_leaf:", rf_model.min_samples_leaf)

# classes and counts
print("classes:", rf_model.classes_)
print("n_classes_:", rf_model.n_classes_)

# feature importances (show top 10)
import numpy as np
fi = rf_model.feature_importances_
print("feature_importances (len={}):".format(len(fi)))
top_idx = np.argsort(fi)[::-1]
for i in top_idx[:10]:
    print(f"  feature {i}: importance={fi[i]:.4f}")


### **pkl** to **onnx** conversion

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Example with 12 features
n_features = 12

initial_type = [("input", FloatTensorType([None, n_features]))]
onnx_model = convert_sklearn(rf_model, initial_types=initial_type)

with open("rf_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())


### ONNX checking integrity against pkl model

In [ ]:
import joblib
import onnxruntime as ort
import numpy as np
from sklearn.metrics import accuracy_score

# === 1. Load your sklearn model (.pkl) ===
rf_model = joblib.load(pkl_model_path)

# === 2. Load the ONNX model ===
sess = ort.InferenceSession("rf_model.onnx")
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

# === 3. Create or load a test set ===
# (here a random example with 12 features)
X_test = np.random.rand(100, 12).astype(np.float32)

# === 4. Prediction from the original model ===
y_pred_sklearn = rf_model.predict(X_test)

# === 5. Prediction from the ONNX model ===
y_pred_onnx = sess.run([output_name], {input_name: X_test})[0]

# Some ONNX models return an array of shape (n, 1) → flatten it
y_pred_onnx = np.squeeze(y_pred_onnx)

# === 6. Comparison ===
equal = np.allclose(y_pred_sklearn, y_pred_onnx, atol=1e-6)
accuracy = accuracy_score(y_pred_sklearn, y_pred_onnx)

print("🔍 Predictions identical:", equal)
print("🎯 Accuracy sklearn vs ONNX:", accuracy)
print("Examples:")
for i in range(5):
    print(f"{i}: sklearn={y_pred_sklearn[i]}, onnx={y_pred_onnx[i]}")


In [ ]:
import joblib

model = joblib.load(pkl_model_path)

print("➡️ Number of features:", model.n_features_in_)

# If the model was trained on a DataFrame:
if hasattr(model, "feature_names_in_"):
    print("➡️ Feature names:", model.feature_names_in_)